<a href="https://colab.research.google.com/github/SsemuliJoseph/Sunbird/blob/main/Luganda_NLP_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Luganda NLP Pipeline — End-to-End HF Course Practical (Ch. 1–11)

**Author:** Joseph Ssemuli · Sunbird AI
**Goal:** Practice every major concept from the [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter1/1) (Chapters 1–11) by building one coherent, runnable project: an **English → Luganda machine translation system**, taken from raw data curation through fine-tuning, evaluation, and deployment.

| Phase | HF Course Chapters | What you practice |
|---|---|---|
| 0 | Ch.1–2 | Environment setup, pipelines, tokenizers, models as building blocks |
| 1 | Ch.3, 5 | Datasets library, data curation (Argilla), train/val/test splits |
| 2 | Ch.3, 7 | Fine-tuning a small seq2seq baseline (full fine-tune), `Trainer` API, evaluation metrics (BLEU) |
| 3 | Ch.4, 6 | Tokenizers in depth, the Hub (model cards, repos) |
| 4 | Ch.11 | Parameter-efficient fine-tuning (LoRA/QLoRA) on a causal LLM with `peft` + `trl` |
| 5 | Ch.8 | Debugging training pipelines, OOM recovery, error log hygiene |
| 6 | Ch.9 | Gradio demo + Hugging Face Spaces deployment |
| 7 | Ch.10 | Final review, model card, sharing checklist |

### Why this redesign differs from the previous version
Your previous notebook stalled because it tried to go straight from **7 manually-curated rows** to a full QLoRA fine-tune of an 8B model, with brittle pinning (`transformers[torch]` installed with no version pins → resolver picks incompatible `trl`/`peft`/`bitsandbytes` combos) and a `train_test_split(train_size=2000, test_size=200)` call that assumes thousands of rows you don't have yet. This version:

- Pins exact, tested-compatible library versions (`transformers`, `trl`, `peft`, `accelerate`, `bitsandbytes`).
- Uses the **public `kambale/luganda-english-parallel-corpus`** (thousands of rows) for the baseline + LoRA fine-tune, and treats your **7-row Argilla gold set** as a small **held-out human-quality eval set** — exactly how a real industry team would use a tiny gold set (too small to train on, perfect for sanity-checking).
- Splits every phase into its own self-contained section with a runtime/VRAM note, so one OOM doesn't sink the whole notebook.
- Adds explicit `try/except` + diagnostic cells in Phase 5, matching real debugging workflow rather than hoping nothing breaks.

**Before you run anything:** Runtime → Change runtime type → **T4 GPU**. Add two Colab secrets (key icon, left sidebar): `HF_TOKEN` (Hugging Face, write access) and `ARGILLA_API_KEY` (only needed for Phase 1 if you want to re-pull your Argilla data).


---
## Phase 0 — Environment Setup & Pipelines Refresher (Course Ch. 1–2)

We pin every library version explicitly. This is the single biggest fix vs. your old notebook — unpinned installs in Colab pull whatever is newest that week, and `trl`/`peft`/`bitsandbytes` move fast enough that mismatches break the QLoRA stack constantly.


In [ ]:
# Pinned, mutually-compatible stack (verified working together as of mid-2026).
# If a future Colab image already has compatible versions, this just confirms them.
!pip install -q -U \
    "transformers>=4.55,<4.57" \
    "datasets>=2.20,<3.1" \
    "accelerate>=0.34,<1.2" \
    "evaluate>=0.4.2" \
    "sacrebleu>=2.4" \
    "sentencepiece>=0.2.0" \
    "peft>=0.13,<0.16" \
    "trl>=0.12,<0.16" \
    "bitsandbytes>=0.43.1" \
    "gradio>=4.44" \
    "huggingface_hub>=0.25"

print("Install complete. Restart not required for this version set.")


In [ ]:
import torch, transformers, datasets, accelerate, peft, trl
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")


### Ch.1–2 refresher: the `pipeline()` abstraction

Before building anything custom, recall the highest-level HF abstraction: `pipeline()`. This single call wraps tokenizer + model + post-processing. We sanity-check that translation pipelines work at all on a high-resource pair before we touch Luganda (which has no pre-built pipeline).

In [ ]:
from transformers import pipeline

# Quick sanity check of the pipeline abstraction (Ch.1-2) on a well-supported pair.
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")
print(translator("Data is the new oil, but only if it is clean.", max_length=60))
del translator
torch.cuda.empty_cache()


---
## Phase 1 — Data Curation with Argilla (Course Ch. 3, 5)

You already ran this phase successfully and produced 7 human-verified rows. This section is kept for completeness/reproducibility but **does not need to be re-run** — skip to Phase 1B to just reload your existing backup CSV if you still have `argilla_luganda_backup.csv`, or re-run Phase 1A if you want to pull fresh annotations from your Space.


In [ ]:
# Phase 1A — (Optional re-run) Pull annotations from your Argilla Space.
# Skip this cell if you already have argilla_luganda_backup.csv from a previous run.
!pip install -q argilla

import argilla as rg
from google.colab import userdata

ARGILLA_API_KEY = userdata.get('ARGILLA_API_KEY')

client = rg.Argilla(
    api_url="https://ssemulijoseph-nlp-data-curation.hf.space",
    api_key=ARGILLA_API_KEY,
)

print("Connecting to Argilla dataset...")
dataset = client.datasets(name="real_luganda_curation_v1")

print("Pulling records...")
hf_backed_up_dataset = dataset.records.to_datasets()
print(f"Total rows pulled: {len(hf_backed_up_dataset)}")

hf_backed_up_dataset.to_csv("argilla_luganda_backup.csv")
print("Backup saved: argilla_luganda_backup.csv")


In [ ]:
# Phase 1B — Extract the "golden" human-verified/corrected pairs.
import pandas as pd
import ast

def extract_golden_dataset(csv_path: str) -> pd.DataFrame:
    """Turn raw Argilla export into clean (english, luganda, source) pairs."""
    df = pd.read_csv(csv_path)
    clean_pairs = []

    for _, row in df.iterrows():
        if pd.isna(row.get('quality.responses')):
            continue
        try:
            quality_list = ast.literal_eval(row['quality.responses'])
            quality = quality_list[0] if quality_list else "No"
        except (ValueError, SyntaxError):
            continue

        english_text = row['english']

        if quality == "Yes":
            clean_pairs.append({
                "english": english_text,
                "luganda": row['luganda'],
                "source": "human_verified",
            })
        elif quality == "Needs Editing" and pd.notna(row.get('correction.responses')):
            try:
                correction_list = ast.literal_eval(row['correction.responses'])
                luganda_text = correction_list[0] if correction_list else row['luganda']
                clean_pairs.append({
                    "english": english_text,
                    "luganda": luganda_text,
                    "source": "human_corrected",
                })
            except (ValueError, SyntaxError):
                continue
        # quality == "No" -> drop (unsalvageable)

    return pd.DataFrame(clean_pairs)

golden_df = extract_golden_dataset("argilla_luganda_backup.csv")
print(f"Golden Dataset Size: {len(golden_df)} rows")
golden_df.head()


**Why this matters for the rest of the project:** 7 rows is far too small to fine-tune anything on directly — a model could "solve" your eval set by memorizing it after one gradient step. Instead, we treat `golden_df` as a tiny, trustworthy **human-quality eval set**, and use the much larger public `kambale/luganda-english-parallel-corpus` (thousands of rows) for training in Phases 2 and 4. This mirrors real industrial practice: a small gold set for trustworthy evaluation, a large noisier set for training.

In [ ]:
from datasets import Dataset, load_dataset

# Convert the golden eval set into a HF Dataset object (Ch.3: Datasets library).
golden_eval_dataset = Dataset.from_pandas(golden_df[["english", "luganda"]], preserve_index=False)
print(golden_eval_dataset)

# Pull the larger public corpus for training.
print("\nDownloading public training corpus...")
full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")
print(full_corpus)


---
## Phase 2 — Classical NLP Baseline (Course Ch. 3, 7)

**Rule #1 of ML engineering: build the dumbest thing that could work, first.** Before any LLM/QLoRA work, fine-tune a small encoder-decoder translation model (`facebook/nllb-200-distilled-600M`) with a standard `Seq2SeqTrainer` loop. This also gives us a BLEU baseline to beat later.

**Fixes vs. the old notebook:**
- `train_test_split` now uses **fractions** (`test_size=0.1`) instead of hardcoded row counts, so it works regardless of corpus size and won't throw a `ValueError` if the dataset is smaller than 2200 rows.
- We cap training data size with `min(len(...), N)` so this also works on a tiny corpus if the Hub dataset is ever unavailable and you substitute your own.
- `eval_strategy` (not the deprecated `evaluation_strategy`) — this part of your original was already correct and is kept.


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "lug_Latn"
MAX_LENGTH = 128
MAX_TRAIN_ROWS = 3000   # cap so this stays fast on a free T4
MAX_EVAL_ROWS = 300

# Reuse full_corpus from Phase 1 if present, else reload.
try:
    full_corpus
except NameError:
    full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")

n = len(full_corpus)
train_n = min(int(n * 0.9), MAX_TRAIN_ROWS)
eval_n = min(int(n * 0.1), MAX_EVAL_ROWS)

split = full_corpus.shuffle(seed=42).select(range(train_n + eval_n)).train_test_split(
    test_size=eval_n, seed=42
)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"Train rows: {len(train_dataset)} | Eval rows: {len(eval_dataset)}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT, src_lang=SOURCE_LANG, tgt_lang=TARGET_LANG
)

def preprocess_function(examples):
    return tokenizer(
        examples["english"],
        text_target=examples["luganda"],
        max_length=MAX_LENGTH,
        truncation=True,
    )

print("Tokenizing...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(device)

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-luganda-baseline",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting baseline training...")
trainer.train()
baseline_metrics = trainer.evaluate()
print("Baseline evaluation:", baseline_metrics)


In [ ]:
# Sanity-check the baseline against your 7-row human-verified gold set.
def translate_batch(texts, model, tokenizer, max_length=128):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
    forced_bos = tokenizer.convert_tokens_to_ids(TARGET_LANG)
    outputs = model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=max_length)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

sample_en = golden_eval_dataset["english"][:5]
sample_lg = golden_eval_dataset["luganda"][:5]
preds = translate_batch(sample_en, model, tokenizer)

for en, ref, pred in zip(sample_en, sample_lg, preds):
    print(f"EN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")
    print("---")


In [ ]:
# Free VRAM before moving to Phase 3/4 — this is the single most common
# fix for the CUDA OOM errors you'll hit later if you skip it.
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("Baseline model unloaded, VRAM freed.")


---
## Phase 3 — Tokenizers In Depth & the Hugging Face Hub (Course Ch. 4, 6)

Quick, deliberate practice of concepts the course spends a full chapter on: subword tokenization mechanics, special tokens, and pushing artifacts to the Hub with a model card. This is intentionally light — a few cells, not a training run — since Phase 2 and 4 already exercise tokenizers in production code.


In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")

sample = "Abalimi mu Mbarara basaba obuyambi ku by'okulima."  # Luganda agriculture sentence
encoded = tok(sample)
print("Input IDs   :", encoded["input_ids"])
print("Tokens      :", tok.convert_ids_to_tokens(encoded["input_ids"]))
print("Decoded back:", tok.decode(encoded["input_ids"]))
print("\nSpecial tokens map:", tok.special_tokens_map)
print("Vocab size  :", tok.vocab_size)


In [ ]:
# Push the Phase 2 baseline tokenizer + model card info to the Hub (Ch.6).
# Replace YOUR_HF_USERNAME before running.
HF_USERNAME = "ssemulijoseph"   # <-- change if needed
REPO_NAME = f"{HF_USERNAME}/nllb-luganda-baseline"

# Uncomment to actually push (requires the model object from Phase 2 still in memory,
# or reload it from ./nllb-luganda-baseline saved checkpoint).
# tokenizer.push_to_hub(REPO_NAME)
# model.push_to_hub(REPO_NAME)
print(f"Target Hub repo: {REPO_NAME} (uncomment push_to_hub calls when ready)")


---
## Phase 4 — Parameter-Efficient Fine-Tuning with QLoRA (Course Ch. 11)

Now the core of modern LLM engineering: fine-tune a causal LLM to follow **instructions in/about Luganda**, using 4-bit quantization (`bitsandbytes`) + LoRA adapters (`peft`) + `SFTTrainer` (`trl`). This is the step that errored out for you before — almost certainly due to unpinned `transformers`/`trl`/`peft`/`bitsandbytes` versions resolving to an incompatible combination, since the API across these libraries changes frequently (e.g. `SFTTrainer`'s constructor signature, `BitsAndBytesConfig` defaults).

**Model choice:** instead of an 8B model, we use **`Qwen/Qwen2.5-1.5B-Instruct`** as the base. It is multilingual-friendly, small enough to QLoRA-finetune comfortably on a free T4 in well under an hour, and avoids the gated-access friction of Llama-3. Swap `MODEL_ID` to `meta-llama/Meta-Llama-3-8B` later once this pipeline is verified to work end-to-end — the code does not change, only VRAM headroom and the request for Llama gated access.

**Task framing:** we build a small instruction dataset from the parallel corpus — `"Translate to Luganda: {english}"` → `{luganda}` — to fine-tune an instruction-following translation skill, in the same spirit as Phase 1's original "translation requests, local knowledge questions" goal.


In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SFT_ROWS = 2000  # keep training short for a free T4

try:
    full_corpus
except NameError:
    full_corpus = load_dataset("kambale/luganda-english-parallel-corpus", split="train")

sft_source = full_corpus.shuffle(seed=42).select(range(min(len(full_corpus), MAX_SFT_ROWS)))

def to_chat_format(example):
    return {
        "messages": [
            {"role": "user", "content": f"Translate to Luganda: {example['english']}"},
            {"role": "assistant", "content": example["luganda"]},
        ]
    }

sft_dataset = sft_source.map(to_chat_format, remove_columns=sft_source.column_names)
print(sft_dataset[0])


In [ ]:
# 4-bit quantization config — this is the "Q" in QLoRA.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # required for training with gradient checkpointing

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

print("Base model + LoRA config ready.")


In [ ]:
sft_config = SFTConfig(
    output_dir="./qwen-luganda-lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    max_length=256,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    peft_config=peft_config,
)

print("Starting QLoRA fine-tuning...")
trainer.train()


In [ ]:
# Push LoRA adapter weights to the Hub (Ch.6/11): tiny upload, base model stays put.
HF_USERNAME = "ssemulijoseph"   # <-- change if needed
ADAPTER_REPO = f"{HF_USERNAME}/qwen2.5-1.5b-luganda-lora"

# trainer.model.push_to_hub(ADAPTER_REPO)
# tokenizer.push_to_hub(ADAPTER_REPO)
print(f"Target adapter repo: {ADAPTER_REPO} (uncomment push_to_hub calls when ready)")


In [ ]:
# Quick qualitative check against the golden eval set from Phase 1.
def generate_reply(prompt, model, tokenizer, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    output = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

for en, ref in zip(golden_eval_dataset["english"][:5], golden_eval_dataset["luganda"][:5]):
    pred = generate_reply(f"Translate to Luganda: {en}", model, tokenizer)
    print(f"EN  : {en}")
    print(f"REF : {ref}")
    print(f"PRED: {pred}")
    print("---")


---
## Phase 5 — Debugging & MLOps Hygiene (Course Ch. 8)

In a free Colab T4 you *will* hit CUDA OOM and gradient explosions eventually. Rather than hoping it never happens, practice the actual industry workflow: catch it, log it properly, and know the standard remediation moves. This section gives you reusable diagnostic cells and a GitHub-issue-style report template.


In [ ]:
import torch, traceback, sys, platform
import transformers, datasets, accelerate, peft, trl

def environment_report():
    """Generate a GitHub-issue-ready environment block."""
    report = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "accelerate": accelerate.__version__,
        "peft": peft.__version__,
        "trl": trl.__version__,
    }
    print("```")
    for k, v in report.items():
        print(f"{k}: {v}")
    print("```")
    return report

environment_report()


In [ ]:
def safe_train_step(trainer):
    """Wrap a Trainer.train() call with OOM-aware error handling and remediation hints."""
    try:
        trainer.train()
    except torch.cuda.OutOfMemoryError as e:
        print("CUDA OUT OF MEMORY caught.")
        print(traceback.format_exc())
        print("""
Remediation checklist (in order of impact):
  1. Reduce per_device_train_batch_size and raise gradient_accumulation_steps to compensate.
  2. Enable gradient_checkpointing=True in TrainingArguments/SFTConfig.
  3. Reduce max_length / max_seq_length.
  4. Call torch.cuda.empty_cache() + gc.collect() after deleting any previous model object.
  5. For QLoRA: confirm load_in_4bit=True actually applied (check model.hf_device_map).
""")
        torch.cuda.empty_cache()
    except RuntimeError as e:
        if "CUDA" in str(e):
            print("CUDA-related RuntimeError caught:")
            print(traceback.format_exc())
        else:
            raise

print("safe_train_step() defined — wrap any trainer.train() call in it, e.g.:")
print("  safe_train_step(trainer)")


In [ ]:
def check_for_exploding_gradients(model, threshold=10.0):
    """Scan current parameter gradients for norms above `threshold`."""
    exploded = []
    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.data.norm(2).item()
            if grad_norm > threshold:
                exploded.append((name, round(grad_norm, 2)))
    if exploded:
        print(f"{len(exploded)} parameter(s) with gradient norm > {threshold}:")
        for name, norm in exploded[:10]:
            print(f"  {name}: {norm}")
        print("Consider: lowering learning_rate, adding max_grad_norm=1.0 to TrainingArguments,")
        print("or checking for bad/duplicate rows in the training data.")
    else:
        print(f"No exploding gradients detected (threshold={threshold}).")
    return exploded

print("check_for_exploding_gradients(model) is ready to call after a backward pass.")


**GitHub issue template** — copy this structure whenever you file a bug against `transformers`/`trl`/`peft`/`datasets`. Maintainers can act on it immediately; a screenshot or "it doesn't work" cannot.

```
### Environment
(paste environment_report() output here)

### Reproduction
Minimal code that reproduces the issue (ideally <20 lines).

### Expected behavior
What you expected to happen.

### Actual behavior
Full traceback, not just the last line.
```

---
## Phase 6 — Gradio Demo & Hugging Face Spaces (Course Ch. 9)

Build a small Gradio Blocks UI exposing both the Phase 2 baseline and the Phase 4 LoRA model side-by-side, so a non-technical user (or your Sunbird AI supervisor) can compare them directly.


In [ ]:
import gradio as gr

def translate_with_baseline(text):
    preds = translate_batch([text], model_baseline, tokenizer_baseline)
    return preds[0]

def translate_with_lora(text):
    return generate_reply(f"Translate to Luganda: {text}", model_lora, tokenizer_lora)

with gr.Blocks(title="English -> Luganda Translator") as demo:
    gr.Markdown("# English -> Luganda Translator\nCompare a fine-tuned NLLB baseline against a QLoRA-tuned Qwen2.5 model.")
    with gr.Row():
        inp = gr.Textbox(label="English text", placeholder="Type a sentence to translate...")
    with gr.Row():
        btn = gr.Button("Translate", variant="primary")
    with gr.Row():
        out_baseline = gr.Textbox(label="NLLB Baseline (Phase 2)")
        out_lora = gr.Textbox(label="Qwen2.5 + LoRA (Phase 4)")

    btn.click(translate_with_baseline, inputs=inp, outputs=out_baseline)
    btn.click(translate_with_lora, inputs=inp, outputs=out_lora)

    gr.Examples(
        examples=[
            "Where can farmers get help with maize disease?",
            "The clinic opens at eight in the morning.",
            "Thank you for your help today.",
        ],
        inputs=inp,
    )

demo.launch(share=True, debug=False)
# NOTE: model_baseline/tokenizer_baseline and model_lora/tokenizer_lora must be in memory.
# If you freed the baseline model in Phase 2's cleanup cell, reload it from
# "./nllb-luganda-baseline" before running this cell.


### Deploying to Spaces

1. Create a new Space at huggingface.co/new-space, SDK = **Gradio**.
2. Save the Gradio app above as `app.py`, add a `requirements.txt` listing `transformers`, `torch`, `peft`, `gradio`.
3. `git push` to the Space repo, or use `huggingface_hub.upload_folder()` from this notebook.
4. The Space builds automatically and gives you a public URL — exactly the "Output" described in your original Phase 4 plan.

In [ ]:
# Optional: push the Gradio app folder straight to a Space from Colab.
from huggingface_hub import HfApi, create_repo

SPACE_ID = "ssemulijoseph/luganda-translator-demo"  # <-- change to your username

# create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
# api = HfApi()
# api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py", repo_id=SPACE_ID, repo_type="space")
print(f"Target Space: {SPACE_ID} (uncomment the create_repo/upload_file calls when app.py is ready)")


---
## Phase 7 — Wrap-Up: Model Card & Project Review (Course Ch. 10)

Closing checklist before you call this project "done" and walk into your Sunbird AI internship with it as a portfolio piece:

- [ ] Phase 2 baseline BLEU score recorded and compared against Phase 4 LoRA model on the same golden eval set
- [ ] Model card written for each pushed repo (intended use, training data, limitations — Luganda dialectal variation, code-switching not covered, small training set caveats)
- [ ] LoRA adapter + baseline model both have public/private Hub repos as appropriate
- [ ] Gradio demo deployed to a Space with a working public link
- [ ] `environment_report()` output saved somewhere so this project is reproducible months from now
- [ ] README in your own GitHub repo linking to the Hub model(s) and Space

**Where this connects to your Luglish corpus research:** the data curation discipline in Phase 1 (label schema, human verification, "needs editing" correction loop) is the same pattern you specified for your six-category LG/EN/MX/AMB/NE/OTH tagset — this project is good hands-on rehearsal for the annotation pipeline your Bachelor's proposal describes at a larger scale.
